# AIMS Senegal — YOLOv11 Fine-Tuning on BDD100K
**Project 2 — Computer Vision | Road Traffic Detection & Tracking**

This notebook fine-tunes YOLOv11n on the BDD100K dataset and displays all training metrics and plots for the report.

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics kagglehub
print("Dependencies installed.")

## 2. GPU Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Enable GPU in Kaggle Settings.")

## 3. Download BDD100K Dataset

In [ ]:
import kagglehub
import os

# Download BDD100K in YOLO format
print("Downloading BDD100K dataset (this may take a few minutes)...")
path = kagglehub.dataset_download("a7madmostafa/bdd100k-yolo")

DATASET_PATH = path
print(f"Dataset path : {DATASET_PATH}")

# Show dataset structure
!ls -lh {DATASET_PATH}
print(f"\nClasses in dataset:")
!cat {DATASET_PATH}/data.yaml

## 4. Fix Dataset YAML Paths

In [ ]:
# BDD100K YAML has hardcoded paths — fix them
original_yaml = f"{DATASET_PATH}/data.yaml"

with open(original_yaml, 'r') as f:
    content = f.read()

content = content.replace('/kaggle/input/bdd100k-yolo', DATASET_PATH)

DATASET_YAML = "/tmp/bdd100k_fixed.yaml"
with open(DATASET_YAML, 'w') as f:
    f.write(content)

print("Fixed YAML content:")
print(content)

## 5. Load Base Model and Show Architecture

In [ ]:
from ultralytics import YOLO

# Load YOLOv11n base model
model = YOLO("yolo11n.pt")

print("Base model loaded : yolo11n.pt")
print(f"Base classes (80 COCO): {list(model.names.values())[:10]}...")
print(f"\nModel info:")
model.info()

## 6. Fine-Tuning YOLOv11 on BDD100K
> Training takes ~20-30 minutes on Kaggle T4 GPU for 10 epochs.

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolo11n.pt")

results = model.train(
    data     = DATASET_YAML,
    epochs   = 10,
    imgsz    = 640,
    batch    = 16,
    workers  = 2,
    device   = 0,
    project  = "runs/train",
    name     = "bdd100k_finetune",
    exist_ok = True,
    patience = 5,
    save     = True,
    plots    = True,
    verbose  = True,
)

print("\nFine-tuning complete !")

## 7. Save Fine-Tuned Model

In [ ]:
import shutil
import glob

# Find best.pt
best_candidates = glob.glob("runs/train/**/best.pt", recursive=True)

if best_candidates:
    src = best_candidates[0]
    os.makedirs("models", exist_ok=True)
    shutil.copy(src, "models/best.pt")
    size_mb = os.path.getsize("models/best.pt") / 1e6
    print(f"Fine-tuned model saved : models/best.pt ({size_mb:.1f} MB)")
    print(f"Source : {src}")
else:
    print("ERROR: best.pt not found. Check training logs above.")

## 8. Verify Fine-Tuned Model Classes

In [ ]:
from ultralytics import YOLO

finetuned = YOLO("models/best.pt")

print("Fine-tuned model classes:")
for idx, name in finetuned.names.items():
    print(f"  {idx} : {name}")

print(f"\nTotal classes : {len(finetuned.names)}")

## 9. Training Metrics — Results CSV

In [ ]:
import pandas as pd
import glob

# Load training results CSV generated by Ultralytics
result_files = glob.glob("runs/train/**/results.csv", recursive=True)

if result_files:
    df = pd.read_csv(result_files[0])
    df.columns = df.columns.str.strip()
    print("Training results (last 5 epochs):")
    print(df.tail())
    print(f"\nBest mAP50   : {df['metrics/mAP50(B)'].max():.4f}")
    print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
else:
    print("results.csv not found.")

## 10. Training Metrics — Plots for Report

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob
import os

result_files = glob.glob("runs/train/**/results.csv", recursive=True)
if not result_files:
    print("No results.csv found. Run training first.")
else:
    df = pd.read_csv(result_files[0])
    df.columns = df.columns.str.strip()
    epochs = df.index + 1

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle("YOLOv11 Fine-Tuning on BDD100K — Training Metrics",
                 fontsize=16, fontweight='bold', y=0.98)

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # 1 — Box Loss (train)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs, df['train/box_loss'], 'b-o', markersize=4)
    ax1.set_title('Train Box Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)

    # 2 — Class Loss (train)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs, df['train/cls_loss'], 'r-o', markersize=4)
    ax2.set_title('Train Class Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.grid(True, alpha=0.3)

    # 3 — DFL Loss (train)
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(epochs, df['train/dfl_loss'], 'g-o', markersize=4)
    ax3.set_title('Train DFL Loss')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Loss')
    ax3.grid(True, alpha=0.3)

    # 4 — Val Box Loss
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(epochs, df['val/box_loss'], 'b--s', markersize=4)
    ax4.set_title('Val Box Loss')
    ax4.set_xlabel('Epoch'); ax4.set_ylabel('Loss')
    ax4.grid(True, alpha=0.3)

    # 5 — Val Class Loss
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.plot(epochs, df['val/cls_loss'], 'r--s', markersize=4)
    ax5.set_title('Val Class Loss')
    ax5.set_xlabel('Epoch'); ax5.set_ylabel('Loss')
    ax5.grid(True, alpha=0.3)

    # 6 — Precision & Recall
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.plot(epochs, df['metrics/precision(B)'], 'purple', marker='o', markersize=4, label='Precision')
    ax6.plot(epochs, df['metrics/recall(B)'],    'orange', marker='s', markersize=4, label='Recall')
    ax6.set_title('Precision & Recall')
    ax6.set_xlabel('Epoch'); ax6.set_ylabel('Score')
    ax6.legend(); ax6.grid(True, alpha=0.3)

    # 7 — mAP50
    ax7 = fig.add_subplot(gs[2, 0])
    ax7.plot(epochs, df['metrics/mAP50(B)'], 'teal', marker='D', markersize=4)
    ax7.set_title('mAP@50')
    ax7.set_xlabel('Epoch'); ax7.set_ylabel('mAP')
    ax7.grid(True, alpha=0.3)

    # 8 — mAP50-95
    ax8 = fig.add_subplot(gs[2, 1])
    ax8.plot(epochs, df['metrics/mAP50-95(B)'], 'darkred', marker='D', markersize=4)
    ax8.set_title('mAP@50-95')
    ax8.set_xlabel('Epoch'); ax8.set_ylabel('mAP')
    ax8.grid(True, alpha=0.3)

    # 9 — All metrics combined
    ax9 = fig.add_subplot(gs[2, 2])
    ax9.plot(epochs, df['metrics/mAP50(B)'],    label='mAP50',   marker='o', markersize=4)
    ax9.plot(epochs, df['metrics/mAP50-95(B)'], label='mAP50-95',marker='s', markersize=4)
    ax9.plot(epochs, df['metrics/precision(B)'],label='Precision',marker='^', markersize=4)
    ax9.plot(epochs, df['metrics/recall(B)'],   label='Recall',  marker='v', markersize=4)
    ax9.set_title('All Metrics Overview')
    ax9.set_xlabel('Epoch'); ax9.set_ylabel('Score')
    ax9.legend(fontsize=8); ax9.grid(True, alpha=0.3)

    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/training_metrics.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart saved : outputs/training_metrics.png")

## 11. Ultralytics Built-in Plots (Confusion Matrix, PR Curve)

In [ ]:
from IPython.display import Image, display
import glob
import os

train_dir = glob.glob("runs/train/bdd100k_finetune")[0] if glob.glob("runs/train/bdd100k_finetune") else None

if train_dir:
    plots = {
        "Confusion Matrix"      : os.path.join(train_dir, "confusion_matrix.png"),
        "Confusion Matrix (norm)": os.path.join(train_dir, "confusion_matrix_normalized.png"),
        "PR Curve"              : os.path.join(train_dir, "PR_curve.png"),
        "F1 Curve"              : os.path.join(train_dir, "F1_curve.png"),
        "Results Overview"      : os.path.join(train_dir, "results.png"),
        "Labels"                : os.path.join(train_dir, "labels.jpg"),
    }
    for title, path in plots.items():
        if os.path.exists(path):
            print(f"\n--- {title} ---")
            display(Image(filename=path, width=800))
        else:
            print(f"Not found : {path}")
else:
    print("Training directory not found. Run training first.")

## 12. Model Validation on Val Set

In [ ]:
from ultralytics import YOLO

model = YOLO("models/best.pt")

print("Running validation on BDD100K val set...")
val_results = model.val(
    data    = DATASET_YAML,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    verbose = True,
)

print(f"\n=== Validation Results ===")
print(f"mAP50    : {val_results.box.map50:.4f}")
print(f"mAP50-95 : {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall   : {val_results.box.mr:.4f}")

## 13. Summary Table for Report

In [ ]:
import pandas as pd
import glob

result_files = glob.glob("runs/train/**/results.csv", recursive=True)
if result_files:
    df = pd.read_csv(result_files[0])
    df.columns = df.columns.str.strip()

    summary = pd.DataFrame({
        "Metric"   : ["mAP@50", "mAP@50-95", "Precision", "Recall"],
        "Best Value": [
            f"{df['metrics/mAP50(B)'].max():.4f}",
            f"{df['metrics/mAP50-95(B)'].max():.4f}",
            f"{df['metrics/precision(B)'].max():.4f}",
            f"{df['metrics/recall(B)'].max():.4f}",
        ],
        "Final Epoch": [
            f"{df['metrics/mAP50(B)'].iloc[-1]:.4f}",
            f"{df['metrics/mAP50-95(B)'].iloc[-1]:.4f}",
            f"{df['metrics/precision(B)'].iloc[-1]:.4f}",
            f"{df['metrics/recall(B)'].iloc[-1]:.4f}",
        ]
    })

    print("=== Summary Table for Report ===")
    print(summary.to_string(index=False))
else:
    print("No results.csv found.")

## 14. List All Output Files

In [ ]:
import os

print("=== models/ ===")
for f in os.listdir("models"):
    size = os.path.getsize(f"models/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")

print("\n=== outputs/ ===")
if os.path.exists("outputs"):
    for f in os.listdir("outputs"):
        print(f"  {f}")

print("\nDone ! Download models/best.pt and push it to your GitHub repo.")